# 🚀 Warmup Cache — Pre-generate Analysis Embeddings

Notebook này chạy trên **Google Colab** để pre-cache tất cả analysis embeddings trước khi training.

**Flow:**
1. Clone repo → Install dependencies
2. Set API key
3. Chạy pipeline cho tất cả windows → cache vào `stock_analysis/st_data/`
4. Training sẽ hit cache, không cần gọi LLM API nữa

**Có thể interrupt bất cứ lúc nào** — đã cache sẽ không gọi lại.

## 1. Setup

In [ ]:
# Clone repo (bỏ qua nếu đã clone)
import os
REPO_URL = "https://github.com/bnquys/stock-trend-prediction.git"
REPO_DIR = "/content/stock-trend-prediction"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print(f"✓ Cloned to {REPO_DIR}")
else:
    !cd {REPO_DIR} && git pull
    print(f"✓ Repo already exists, pulled latest")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
!pip install -q numpy pandas matplotlib scipy scikit-learn pyyaml torch requests python-dotenv tabulate

In [ ]:
# ⚠️ QUAN TRỌNG: Set API key cho LLM
# Cách 1: Nhập trực tiếp (không khuyến khích cho repo public)
# Cách 2: Dùng Colab Secrets (khuyến khích)

import os
from pathlib import Path

# --- Cách 1: Dùng Google Colab Secrets (nếu đã thêm secret 'API_KEY') ---
try:
    from google.colab import userdata
    api_key = userdata.get('API_KEY')
    print("✓ Loaded API_KEY from Colab Secrets")
except Exception:
    # --- Cách 2: Nhập thủ công ---
    from getpass import getpass
    api_key = getpass("Nhập API_KEY: ")
    print("✓ API_KEY set manually")

# Ghi vào .env cho stock_analysis
env_path = Path("stock_analysis/.env")
env_path.write_text(f"API_KEY={api_key}\n")
os.environ["API_KEY"] = api_key
print(f"✓ Written to {env_path}")

## 2. Load Config & Kiểm tra Data

In [ ]:
import sys, logging, time
import yaml
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

sys.path.insert(0, '.')

from src.features.preprocessor import load_csv
from stock_analysis import pipeline

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s', datefmt='%H:%M:%S')

# Load config
with open('configs/config.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

WINDOW = cfg['env']['window']  # 20
MODEL = cfg['analysis']['model']
PATHS = cfg['data']['paths']

print(f'Window: {WINDOW}')
print(f'Model:  {MODEL}')
print(f'Stocks: {[Path(p).stem for p in PATHS]}')
print()

# Kiểm tra data files
for p in PATHS:
    if Path(p).exists():
        df = load_csv(p)
        print(f"  ✓ {Path(p).stem}: {len(df)} rows | {df['date'].iloc[0].date()} → {df['date'].iloc[-1].date()} | Windows: {len(df)-WINDOW}")
    else:
        print(f"  ✗ {p} NOT FOUND!")

## 3. Chạy Cache

Mỗi window gọi pipeline 1 lần. Pipeline tự cache theo hash nội dung report:
- Nếu report đã tồn tại → skip tạo report
- Nếu LLM response đã tồn tại → skip gọi API
- Nếu embedding đã tồn tại → skip tính embedding

→ Chạy lại an toàn, chỉ tốn thời gian cho windows chưa cache.

In [ ]:
def warmup_stock(stock_id: str, csv_path: str, window: int = 20, skip_every: int = 1):
    """
    Cache tất cả windows cho 1 stock.
    
    Args:
        skip_every: Cache mỗi N windows (1 = tất cả, 5 = mỗi 5 ngày).
                    Dùng skip_every > 1 nếu muốn nhanh hơn (trade-off: một số step sẽ không có embedding).
    """
    df = load_csv(csv_path)
    total = len(df) - window
    n_to_cache = total // skip_every
    cached = 0
    errors = 0
    start_t = time.time()
    
    print(f'\n{"═"*60}')
    print(f'  Caching {stock_id}: {n_to_cache} windows (skip_every={skip_every})')
    print(f'{"═"*60}')
    
    for i, step in enumerate(range(window, len(df), skip_every)):
        start_idx = max(0, step - window + 1)
        date_end = pd.Timestamp(df['date'].iloc[step]).to_pydatetime()
        date_start = pd.Timestamp(df['date'].iloc[start_idx]).to_pydatetime()
        
        try:
            result = pipeline(
                model=MODEL,
                stock_id=stock_id,
                date_start=date_start,
                date_end=date_end,
            )
            cached += 1
            
            # Progress log mỗi 50 windows
            if cached % 50 == 0:
                elapsed = time.time() - start_t
                rate = cached / elapsed
                remaining = (n_to_cache - cached) / max(rate, 0.01)
                print(f'  [{stock_id}] {cached}/{n_to_cache} '
                      f'({cached*100//n_to_cache}%) | '
                      f'{rate:.1f} win/s | ETA: {remaining/60:.1f}m')
                
        except Exception as e:
            errors += 1
            if errors <= 5:
                print(f'  [ERROR] step={step}, date={date_end.date()}: {e}')
            elif errors == 6:
                print(f'  ... (suppressing further errors)')
    
    elapsed = time.time() - start_t
    print(f'\n  ✓ {stock_id} done: {cached} cached, {errors} errors, {elapsed/60:.1f} min')
    return cached, errors

### Chạy từng stock (có thể chạy riêng từng cell, interrupt an toàn)

In [ ]:
warmup_stock('VNM', 'data/VNM.csv', WINDOW, skip_every=1)

In [ ]:
warmup_stock('FPT', 'data/FPT.csv', WINDOW, skip_every=1)

In [ ]:
warmup_stock('VIC', 'data/VIC.csv', WINDOW, skip_every=1)

In [ ]:
warmup_stock('HPG', 'data/HPG.csv', WINDOW, skip_every=1)

## 4. Kiểm tra Cache

In [ ]:
st_data = Path('stock_analysis/st_data')
print(f'{"Stock":<8} {"Reports":>10} {"Responses":>12} {"Embeddings":>12}')
print('─' * 45)
for stock_dir in sorted(st_data.iterdir()):
    if not stock_dir.is_dir():
        continue
    reports = list((stock_dir / 'reports').glob('*.md')) if (stock_dir / 'reports').exists() else []
    responses = list((stock_dir / 'responses').glob('*.md')) if (stock_dir / 'responses').exists() else []
    embeddings = list((stock_dir / 'responses').glob('*.npy')) if (stock_dir / 'responses').exists() else []
    print(f'{stock_dir.name:<8} {len(reports):>10} {len(responses):>12} {len(embeddings):>12}')

## 5. Train (Optional — chạy luôn trên Colab)

Sau khi cache xong, có thể train ngay:

In [ ]:
# Chạy training (thay đổi --episodes theo nhu cầu)
!python train.py --episodes 500